In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.ticker import FuncFormatter, LogLocator, LogFormatterMathtext

counts_file = "/Users/apple/Downloads/bacdrops/sal/counts.tsv"
barcode_file = "/Users/apple/Downloads/bacdrops/sal/run1.barcodes.tsv"
# Number of top-ranked barcodes
N_TOP = 50000
# Number of linear barcode-rank bins
N_RANK_BINS = 50
# LOWESS smoothing parameter
LOWESS_FRAC = 0.03

counts = pd.read_csv(counts_file, sep="\t")

counts = counts.sort_values("umi_count", ascending=False).reset_index(drop=True)

counts["barcode_rank"] = np.arange(len(counts)) + 1

n_top = min(N_TOP, len(counts))
plot_df = counts.iloc[:n_top].copy()

bc = pd.read_csv(barcode_file, sep="\t")

call_info = bc[["barcode", "call"]].drop_duplicates(subset="barcode")

plot_df = plot_df.merge(call_info, on="barcode", how="left")

plot_df["is_cell"] = plot_df["call"] == "cell"

plot_df = plot_df[plot_df["umi_count"] > 0].copy()
# 10. LOWESS
# x = linear barcode rank
# y = actual UMI count
# IMPORTANT:
# We do NOT log-transform UMI before LOWESS.
# The logarithmic appearance comes from the y-axis scale.
lowess_fit = lowess(
    endog=plot_df["umi_count"], exog=plot_df["barcode_rank"], frac=LOWESS_FRAC, it=2, return_sorted=True
)
lowess_x = lowess_fit[:, 0]
lowess_y = lowess_fit[:, 1]
# 11. Linear barcode-rank bins
# 50 bins across 0–50,000
# Each bin contains approximately 1,000 barcodes.
plot_df["rank_bin"] = pd.cut(plot_df["barcode_rank"], bins=N_RANK_BINS, include_lowest=True)
# 12. Calculate cell fraction
# Fraction = number of call == cell
#              all barcodes
# within each barcode-rank bin.
fraction_df = (
    plot_df.groupby("rank_bin", observed=True)
    .agg(barcode_rank=("barcode_rank", "mean"), cell_fraction=("is_cell", "mean"), n_barcodes=("is_cell", "size"))
    .reset_index(drop=True)
)

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

fig, ax = plt.subplots(figsize=(4, 3))

ax.plot(lowess_x, lowess_y, linewidth=1.0, color="black", zorder=4, label="LOWESS")
# 16. X axis
# Linear barcode rank
# 0  1  2  3  4  5
# corresponding to
# 0  1  2  3  4  5 × 10^4
ax.set_xlim(0, N_TOP)
ax.set_xticks(np.arange(0, N_TOP + 1, 10000))
def rank_formatter(x, pos):
    return f"{x / 1e4:g}"
ax.xaxis.set_major_formatter(FuncFormatter(rank_formatter))
ax.set_xlabel(r"Barcode rank ($\times 10^4$)")

ax.set_yscale("log")
ax.set_ylim(50, 1e5)
ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0,)))
ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
ax.set_ylabel("UMI counts")

ax2 = ax.twinx()
# 19. Cell fraction bars
# Use exact bin edges so there are NO gaps.
bin_edges = np.linspace(0, N_TOP, N_RANK_BINS + 1)
ax2.bar(
    bin_edges[:-1],
    fraction_df["cell_fraction"],
    width=np.diff(bin_edges),
    align="edge",
    alpha=0.30,
    linewidth=0,
    edgecolor="none",
    zorder=1,
    label="Cell fraction",
)

ax2.set_ylabel("Fraction called as cell")
ax2.set_ylim(0, 1)
ax2.set_yticks(np.arange(0.2, 1.01, 0.2))
ax2.set_yticklabels([f"{x:.1f}" for x in np.arange(0.2, 1.01, 0.2)])

ax.grid(axis="both", alpha=0.15, linewidth=0.6, zorder=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="lower left", frameon=False, handlelength=2.5)

fig.tight_layout()


plt.show()
plt.close(fig)
#note4fig: fig2a_part2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.ticker import FuncFormatter, LogLocator, LogFormatterMathtext

counts_file = "/Users/apple/Downloads/bacdrops/sal/counts.tsv"
# Number of top-ranked barcodes to plot
N_TOP = 50000
# Number of linear barcode-rank bins
N_RANK_BINS = 50
# Number of called barcodes
N_CALLED = 5770
# LOWESS smoothing
LOWESS_FRAC = 0.03
# Box line width
BOX_LINEWIDTH = 0.8

counts = pd.read_csv(counts_file, sep="\t")

counts = counts.sort_values("umi_count", ascending=False).reset_index(drop=True)

counts["barcode_rank"] = np.arange(len(counts)) + 1

n_top = min(N_TOP, len(counts))
plot_df = counts.iloc[:n_top].copy()
# 5. Define calling status
# IMPORTANT:
# No run1.barcodes.tsv is used.
# The first 5770 barcodes are called cells.
plot_df["is_cell"] = plot_df["barcode_rank"] <= N_CALLED

plot_df = plot_df[plot_df["umi_count"] > 0].copy()
# 7. LOWESS
# x = linear barcode rank
# y = actual UMI count
# No log transformation before LOWESS.
lowess_fit = lowess(
    endog=plot_df["umi_count"], exog=plot_df["barcode_rank"], frac=LOWESS_FRAC, it=2, return_sorted=True
)
lowess_x = lowess_fit[:, 0]
lowess_y = lowess_fit[:, 1]

plot_df["rank_bin"] = pd.cut(plot_df["barcode_rank"], bins=N_RANK_BINS, include_lowest=True)
# 9. Calculate cell fraction
# Since the bins are based on linear barcode rank,
# each bin contains approximately the same number
# of barcodes.
fraction_df = (
    plot_df.groupby("rank_bin", observed=True)
    .agg(barcode_rank=("barcode_rank", "mean"), cell_fraction=("is_cell", "mean"), n_barcodes=("is_cell", "size"))
    .reset_index(drop=True)
)

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

fig, ax = plt.subplots(figsize=(4, 3))

ax.plot(lowess_x, lowess_y, linewidth=1.0, color="black", zorder=4, label="LOWESS")

ax.set_xlim(0, N_TOP)
ax.set_xticks(np.arange(0, N_TOP + 1, 10000))
def rank_formatter(x, pos):
    return f"{x / 1e4:g}"
ax.xaxis.set_major_formatter(FuncFormatter(rank_formatter))
ax.set_xlabel(r"Barcode rank ($\times 10^4$)")

ax.set_yscale("log")
ax.set_ylim(50, 1e5)
ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0,)))
ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
ax.set_ylabel("UMI counts")

ax2 = ax.twinx()
# 16. Cell fraction bars
# Exact bin edges → NO gaps between bars.
bin_edges = np.linspace(0, N_TOP, N_RANK_BINS + 1)
ax2.bar(
    bin_edges[:-1],
    fraction_df["cell_fraction"],
    width=np.diff(bin_edges),
    align="edge",
    alpha=0.30,
    linewidth=0,
    edgecolor="none",
    zorder=1,
    label="Cell fraction",
)

ax2.set_ylabel("Fraction called as cell")
ax2.set_ylim(0, 1)
ax2.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])

ax.grid(axis="both", alpha=0.15, linewidth=0.6, zorder=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(BOX_LINEWIDTH)
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(BOX_LINEWIDTH)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="lower left", frameon=False, handlelength=2.5)

fig.tight_layout()


plt.show()
plt.close(fig)
#note4fig: fig2a_part1

In [ ]:
import pandas as pd
import re

gene_map = []
with open("/Users/apple/Downloads/bacdrops/sal/28901.gff") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if len(cols) != 9:
            continue
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        gene_id = None
        gene_name = None
        m = re.search(r"ID=([^;]+)", attrs)
        if m:
            gene_id = m.group(1)
        m = re.search(r"gene=([^;]+)", attrs)
        if m:
            gene_name = m.group(1)
        else:
            m = re.search(r"Name=([^;]+)", attrs)
            if m:
                gene_name = m.group(1)
        if gene_id is not None:
            gene_map.append([gene_id, gene_name])
gene_map = pd.DataFrame(gene_map, columns=["names", "gene_name"])
print(gene_map.head())

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
adata = sc.read_h5ad("md.hvg.h5ad")

# Recompute UMAP layout (keep kNN graph → leiden unchanged)
# Original: min_dist=0.5 (default). Recommended: min_dist=0.8 + negative_sample_rate=20
# Reduces spiky/filamentous structures by ~43% (test_umap_layout.ipynb + project_memory.md)
sc.tl.umap(adata, min_dist=0.8, negative_sample_rate=20, random_state=0)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

sc.tl.leiden(adata, resolution=0.3)
print(adata.obs["leiden"].value_counts())

sc.pl.umap(adata, color="leiden", legend_loc="on data", show=False)

ax = plt.gca()
for collection in ax.collections:
    collection.set_rasterized(True)

plt.close()
#note4fig: fig2c_part2

In [ ]:
adata_full = sc.read_h5ad("md.full.h5ad")

adata_full.obs["leiden"] = adata.obs["leiden"].reindex(adata_full.obs_names)
print(adata_full.obs["leiden"].value_counts(dropna=False))
# 8. Find marker genes using Leiden from HVG-based adata
# but test against all genes in adata_full
sc.tl.rank_genes_groups(adata_full, groupby="leiden", method="wilcoxon", key_added="rank_genes_leiden")
markers = sc.get.rank_genes_groups_df(adata_full, group=None, key="rank_genes_leiden")
markers = markers.merge(gene_map, on="names", how="left")
markers["marker"] = markers["gene_name"].fillna(markers["names"])
markers.to_csv("leiden_markers.tsv", sep="\t", index=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

manual_gene_ids = ['gapA', 'ompC', 'fimA', 'tktB', 'thrA', 'pta', 'sdaC', 'nlpI', 'ymdF', 'dps', 'rpoS', 'yfiA']

gene_map_dict = dict(zip(gene_map["gene_name"], gene_map["names"]))
plot_gene_ids = []
gene_names = []
for gene in manual_gene_ids:
    if gene not in gene_map_dict:
        print(f"WARNING: {gene} not found in gene_map")
        continue
    gid = gene_map_dict[gene]
    if gid not in adata_full.var_names:
        print(f"WARNING: {gene} -> {gid}, " f"but not found in adata_full")
        continue
    plot_gene_ids.append(gid)
    gene_names.append(gene)

print("\nGenes used for bubble plot:")
for gene, gid in zip(gene_names, plot_gene_ids):
    print(f"{gene:10s} -> {gid}")

X = adata_full[:, plot_gene_ids].X
if hasattr(X, "toarray"):
    X = X.toarray()
expr = pd.DataFrame(X, index=adata_full.obs_names, columns=gene_names)

expr["cluster"] = adata_full.obs["leiden"].astype(str).values

bubble = []
for cluster, group in expr.groupby("cluster"):
    for gene in gene_names:
        values = group[gene]
        bubble.append(
            {
                "cluster": cluster,
                "gene": gene,
                # fraction of cells expressing
                "fraction": (values > 0).mean(),
                # mean expression across all cells
                "mean_expr": values.mean(),
            }
        )
bubble = pd.DataFrame(bubble)
# 7. Reference expression
# Minimum mean expression across clusters
ref_expr = bubble.groupby("gene")["mean_expr"].min()

pseudocount = 1e-3
bubble["log2FC"] = np.log2((bubble["mean_expr"] + pseudocount) / (bubble["gene"].map(ref_expr) + pseudocount))

clusters = sorted(bubble["cluster"].unique(), key=lambda x: int(x))
genes = gene_names
x_map = {g: i for i, g in enumerate(genes)}
y_map = {c: i for i, c in enumerate(clusters)}

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

bubble_size = (bubble["fraction"] ** 0.8) * 400
scat = ax.scatter(
    bubble["gene"].map(x_map),
    bubble["cluster"].map(y_map),
    s=bubble_size,
    # Color = log2FC
    c=bubble["log2FC"],
    cmap="RdBu_r",
    vmin=0,
    vmax=10,
    edgecolor="black",
    linewidth=0.25,
)

ax.set_xticks(range(len(genes)))
ax.set_xticklabels(genes, rotation=45, ha="right")
ax.set_yticks(range(len(clusters)))
ax.set_yticklabels(["Cluster " + c for c in clusters])
ax.set_ylabel("Leiden cluster")

cbar = plt.colorbar(scat, ax=ax)
cbar.set_label("log$_2$ fold-change\n(vs lowest-expression cluster)")

for frac in [0.25, 0.5, 0.75, 1.0]:
    ax.scatter([], [], s=(frac**0.8) * 400, color="grey", edgecolor="black", linewidth=0.25, label=f"{frac:.0%}")
ax.legend(title="Expressing cells", bbox_to_anchor=(1.35, 1), frameon=False)

for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()

plt.show()
#note4fig: fig2d_part2

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

adata = sc.read_h5ad("md.hvg.h5ad")

# Recompute UMAP layout (keep kNN graph → leiden unchanged)
# Original: min_dist=0.5 (default). Recommended: min_dist=0.8 + negative_sample_rate=20
# Reduces spiky/filamentous structures by ~43% (test_umap_layout.ipynb + project_memory.md)
sc.tl.umap(adata, min_dist=0.8, negative_sample_rate=20, random_state=0)
print(adata)
print(adata.var.columns)

gene_dict = dict(zip(gene_map["names"], gene_map["gene_name"]))
adata.var["gene_name"] = [gene_dict.get(gene_id, gene_id) for gene_id in adata.var_names]

genes = ["rpoS", "fimA", "pta"]

gene_to_id = dict(zip(adata.var["gene_name"], adata.var_names))
genes_present = [gene for gene in genes if gene in gene_to_id]
gene_ids = [gene_to_id[gene] for gene in genes_present]
print("Genes found:")
print(genes_present)
print("\nGenes not found:")
print([gene for gene in genes if gene not in gene_to_id])

sc.pl.umap(
    adata,
    color=gene_ids,
    ncols=3,
    use_raw=False,
    cmap="viridis",
    vmin=0,
    vmax="p99",
    size=12,
    frameon=False,
    show=False,
)

# Scanpy creates one or more PathCollection objects
# for the cell points.
# Rasterize only these collections.
# Text / colorbars remain vector.
for ax in plt.gcf().axes:
    for collection in ax.collections:
        collection.set_rasterized(True)

plt.tight_layout()
plt.show()
plt.close()
#note4fig: fig2h_part2 fig2i_part2 fig2j_part2

In [ ]:
module_genes = {
    # 1. Translation / Ribosome
    # Protein synthesis / growth-associated programme
    "Translation": [
        # 50S
        "rplA", "rplB", "rplC", "rplD", "rplE",
        "rplF", "rplG", "rplH", "rplI", "rplJ",
        "rplK", "rplL", "rplM", "rplN", "rplO",
        "rplP", "rplQ", "rplR", "rplS", "rplT",
        "rplU", "rplV", "rplW", "rplX", "rplY",
        # 30S
        "rpsA", "rpsB", "rpsC", "rpsD", "rpsE",
        "rpsF", "rpsG", "rpsH", "rpsI", "rpsJ",
        "rpsK", "rpsL", "rpsM", "rpsN", "rpsO",
        "rpsP", "rpsQ", "rpsR", "rpsS", "rpsT",
        # Translation factors
        "tufA", "fusA", "tsf", "infA", "infB", "infC",
        "lepA", "typA", "tufB"
    ],
    "Replication": [
        "dnaA", "dnaB", "dnaC", "dnaE", "dnaG",
        "dnaN", "dnaQ", "dnaT", "dnaX",
        "gyrA", "gyrB",
        "parC", "parE",
        "recA", "recF", "recN",
        "holA", "holB", "holC", "holD",
        "priA", "priB", "priC",
        "ssb",
        "dam", "mutS", "mutL"
    ],
    # 3. Central metabolism
    # Glycolysis / TCA / PPP / carbon metabolism
    "Metabolism": [
        # Glycolysis
        "gapA", "pgk", "gpmA", "eno", "pykA",
        "pfkA", "pfkB", "fbaA", "pgi", "zwf",
        # Pentose phosphate pathway
        "tktA", "tktB", "talA", "talB",
        "gnd",
        # TCA
        "gltA", "acnA", "acnB",
        "icdA",
        "sucA", "sucB", "sucC", "sucD",
        "sdhA", "sdhB", "sdhC", "sdhD",
        "fumA", "fumB",
        "mdh",
        # Pyruvate / acetyl-CoA
        "aceE", "aceF", "lpdA",
        "pdhR",
        "pta", "ackA",
        # PTS / carbon uptake
        "ptsI", "ptsH",
        "crr",
        # Acetate / fermentation
        "adhE", "ldhA", "pflB"
    ],
    # 4. Stress / RpoS
    # General stress / stationary-phase-associated programme
    "Stress_RpoS": [
        "rpoS",
        # Oxidative stress
        "dps",
        "katG", "katE",
        "ahpC", "ahpF",
        "sodA", "sodB",
        # General stress / stationary phase
        "osmY",
        "osmC",
        "yciF", "yciG", "yciE",
        "hchA",
        "ibpA", "ibpB",
        # Protein folding / proteostasis
        "dnaK", "dnaJ",
        "groL", "groS",
        "hslU", "hslV",
        # Stationary-phase / translation downshift
        "yfiA",
        "uspA",
        "uspB",
        "uspC",
        "uspD",
        "uspE"
    ],
    "Fimbrial": [
        "fimA",
        "fimI",
        "fimC",
        "fimD",
        "fimF",
        "fimG",
        "fimH"
    ],
    # 6. Adhesion / colonisation
    # Surface adhesion / intestinal persistence-associated
    "Adhesion": [
        # Type 1 fimbriae
        "fimA", "fimI", "fimC", "fimD",
        "fimF", "fimG", "fimH",
        # CS54 / intestinal persistence-associated
        "shdA",
        "ratA",
        "ratB",
        # Other adhesins / surface-associated factors
        "misL",
        "bigA",
        "sinH",
        "stdA",
        "stdB",
        "stdC",
        "stdD",
        "stdE",
        "stdF",
        "stdG",
        "stdH"
    ],
    "Flagellar": [
        # Basal body / hook
        "flhA", "flhB", "flhC", "flhD",
        "flgA", "flgB", "flgC", "flgD",
        "flgE", "flgF", "flgG", "flgH",
        "flgI", "flgJ", "flgK", "flgL",
        # Motor
        "motA", "motB",
        # Filament
        "fliC",
        "fliD",
        "fliK",
        "fliT",
        # Flagellar assembly / regulation
        "fliA",
        "fliZ",
        "fliS",
        "fliY",
        "fliE",
        "fliF",
        "fliG",
        "fliH",
        "fliI",
        "fliJ",
        "fliM",
        "fliN",
        "fliO",
        "fliP",
        "fliQ",
        "fliR"
    ],
    "Chemotaxis": [
        "cheA",
        "cheB",
        "cheR",
        "cheW",
        "cheY",
        "cheZ",
        # Methyl-accepting chemotaxis proteins
        "tar",
        "tap",
        "tsr",
        "trg"
    ],
    "SPI1": [
        # Main regulatory system
        "hilA",
        "hilC",
        "hilD",
        "invF",
        # Type III secretion apparatus
        "invA",
        "invB",
        "invC",
        "invE",
        "invG",
        "invH",
        "spaP",
        "spaQ",
        "spaR",
        "spaS",
        # Secreted effectors
        "sipA",
        "sipB",
        "sipC",
        "sipD",
        "sicA",
        "sicP",
        "sptP",
        "avrA"
    ],
    "SPI2": [
        # Regulatory system
        "ssrA",
        "ssrB",
        # T3SS apparatus
        "ssaB",
        "ssaC",
        "ssaD",
        "ssaE",
        "ssaF",
        "ssaG",
        "ssaH",
        "ssaI",
        "ssaJ",
        "ssaK",
        "ssaL",
        "ssaM",
        "ssaN",
        "ssaO",
        "ssaP",
        "ssaQ",
        "ssaR",
        "ssaS",
        "ssaT",
        "ssaU",
        "ssaV",
        # Effectors
        "sseB",
        "sseC",
        "sseD",
        "sseE",
        "sseF",
        "sseG",
        "sseI",
        "sseJ",
        "sifA",
        "sifB"
    ],
    "Iron_Acquisition": [
        # Enterobactin synthesis
        "entA",
        "entB",
        "entC",
        "entD",
        "entE",
        "entF",
        # Enterobactin transport
        "fepA",
        "fepB",
        "fepC",
        "fepD",
        "fepG",
        # Ferric uptake / siderophore systems
        "fhuA",
        "fhuB",
        "fhuC",
        "fhuD",
        # Sit system
        "sitA",
        "sitB",
        "sitC",
        "sitD",
        # Iron regulation
        "fur"
    ]
}

In [ ]:
import pandas as pd
import re

gene_map = []
with open("/Users/apple/Downloads/bacdrops/sal/28901.gff") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if len(cols) != 9:
            continue
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        gene_id = None
        gene_name = None
        m = re.search(r"ID=([^;]+)", attrs)
        if m:
            gene_id = m.group(1)
        m = re.search(r"gene=([^;]+)", attrs)
        if m:
            gene_name = m.group(1)
        else:
            m = re.search(r"Name=([^;]+)", attrs)
            if m:
                gene_name = m.group(1)
        if gene_id is not None:
            gene_map.append([gene_id, gene_name])
gene_map = pd.DataFrame(gene_map, columns=["names", "gene_name"])
print(gene_map.head())

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
adata = sc.read_h5ad("/Users/apple/Downloads/notebook/knee.hvg.h5ad")

# Recompute UMAP layout (keep kNN graph → leiden unchanged)
# Original: min_dist=0.5 (default). Recommended: min_dist=0.8 + negative_sample_rate=20
# Reduces spiky/filamentous structures by ~43% (test_umap_layout.ipynb + project_memory.md)
sc.tl.umap(adata, min_dist=0.8, negative_sample_rate=20, random_state=0)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

sc.tl.leiden(adata, resolution=0.4)

min_cells = 50
cluster_counts = adata.obs["leiden"].value_counts()
small_clusters = cluster_counts[cluster_counts < min_cells].index
print("Cluster sizes:")
print(cluster_counts.sort_index())
print("\nSmall clusters:")
print(cluster_counts[cluster_counts < min_cells])
adata.obs["leiden_filtered"] = adata.obs["leiden"].astype(str)
adata.obs.loc[adata.obs["leiden_filtered"].isin(small_clusters), "leiden_filtered"] = "small"
print("\nFinal clusters:")
print(adata.obs["leiden_filtered"].value_counts())

sc.pl.umap(adata, color="leiden_filtered", legend_loc="on data", show=False)
ax = plt.gca()
for collection in ax.collections:
    collection.set_rasterized(True)
plt.close()
#note4fig: fig2c_part1

In [ ]:
adata_full = sc.read_h5ad("/Users/apple/Downloads/notebook/knee.full.h5ad")

adata_full.obs["leiden"] = adata.obs["leiden"].reindex(adata_full.obs_names)
print(adata_full.obs["leiden"].value_counts(dropna=False))
# 8. Find marker genes using Leiden from HVG-based adata
# but test against all genes in adata_full
sc.tl.rank_genes_groups(adata_full, groupby="leiden", method="wilcoxon", key_added="rank_genes_leiden")
markers = sc.get.rank_genes_groups_df(adata_full, group=None, key="rank_genes_leiden")
markers = markers.merge(gene_map, on="names", how="left")
markers["marker"] = markers["gene_name"].fillna(markers["names"])
markers.to_csv("leiden_markers.tsv", sep="\t", index=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

manual_gene_ids = ['ymdF', 'fimA', 'fimC', 'fimI']

gene_map_dict = dict(zip(gene_map["gene_name"], gene_map["names"]))
plot_gene_ids = []
gene_names = []
for gene in manual_gene_ids:
    if gene not in gene_map_dict:
        print(f"WARNING: {gene} not found in gene_map")
        continue
    gid = gene_map_dict[gene]
    if gid not in adata_full.var_names:
        print(f"WARNING: {gene} -> {gid}, " f"but not found in adata_full")
        continue
    plot_gene_ids.append(gid)
    gene_names.append(gene)

print("\nGenes used for bubble plot:")
for gene, gid in zip(gene_names, plot_gene_ids):
    print(f"{gene:10s} -> {gid}")

X = adata_full[:, plot_gene_ids].X
if hasattr(X, "toarray"):
    X = X.toarray()
expr = pd.DataFrame(X, index=adata_full.obs_names, columns=gene_names)

expr["cluster"] = adata_full.obs["leiden"].astype(str).values

bubble = []
for cluster, group in expr.groupby("cluster"):
    for gene in gene_names:
        values = group[gene]
        bubble.append(
            {
                "cluster": cluster,
                "gene": gene,
                # fraction of cells expressing
                "fraction": (values > 0).mean(),
                # mean expression across all cells
                "mean_expr": values.mean(),
            }
        )
bubble = pd.DataFrame(bubble)
# 7. Reference expression
# Minimum mean expression across clusters
ref_expr = bubble.groupby("gene")["mean_expr"].min()

pseudocount = 1e-3
bubble["log2FC"] = np.log2((bubble["mean_expr"] + pseudocount) / (bubble["gene"].map(ref_expr) + pseudocount))

clusters = sorted(bubble["cluster"].unique(), key=lambda x: int(x))
genes = gene_names
x_map = {g: i for i, g in enumerate(genes)}
y_map = {c: i for i, c in enumerate(clusters)}

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

bubble_size = (bubble["fraction"] ** 0.8) * 400
scat = ax.scatter(
    bubble["gene"].map(x_map),
    bubble["cluster"].map(y_map),
    s=bubble_size,
    # Color = log2FC
    c=bubble["log2FC"],
    cmap="RdBu_r",
    vmin=0,
    vmax=10,
    edgecolor="black",
    linewidth=0.25,
)

ax.set_xticks(range(len(genes)))
ax.set_xticklabels(genes, rotation=45, ha="right")
ax.set_yticks(range(len(clusters)))
ax.set_yticklabels(["Cluster " + c for c in clusters])
ax.set_ylabel("Leiden cluster")

cbar = plt.colorbar(scat, ax=ax)
cbar.set_label("log$_2$ fold-change\n(vs lowest-expression cluster)")

for frac in [0.25, 0.5, 0.75, 1.0]:
    ax.scatter([], [], s=(frac**0.8) * 400, color="grey", edgecolor="black", linewidth=0.25, label=f"{frac:.0%}")
ax.legend(title="Expressing cells", bbox_to_anchor=(1.35, 1), frameon=False)

for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()

plt.show()
#note4fig: fig2d_part1

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

adata = sc.read_h5ad("knee.hvg.h5ad")

# Recompute UMAP layout (keep kNN graph → leiden unchanged)
# Original: min_dist=0.5 (default). Recommended: min_dist=0.8 + negative_sample_rate=20
# Reduces spiky/filamentous structures by ~43% (test_umap_layout.ipynb + project_memory.md)
sc.tl.umap(adata, min_dist=0.8, negative_sample_rate=20, random_state=0)
print(adata)
print(adata.var.columns)

gene_dict = dict(zip(gene_map["names"], gene_map["gene_name"]))
adata.var["gene_name"] = [gene_dict.get(gene_id, gene_id) for gene_id in adata.var_names]

genes = ["rpoS", "fimA", "pta"]

gene_to_id = dict(zip(adata.var["gene_name"], adata.var_names))
genes_present = [gene for gene in genes if gene in gene_to_id]
gene_ids = [gene_to_id[gene] for gene in genes_present]
print("Genes found:")
print(genes_present)
print("\nGenes not found:")
print([gene for gene in genes if gene not in gene_to_id])

sc.pl.umap(
    adata,
    color=gene_ids,
    ncols=3,
    use_raw=False,
    cmap="viridis",
    vmin=0,
    vmax="p99",
    size=12,
    frameon=False,
    show=False,
)

# Scanpy creates one or more PathCollection objects
# for the cell points.
# Rasterize only these collections.
# Text / colorbars remain vector.
for ax in plt.gcf().axes:
    for collection in ax.collections:
        collection.set_rasterized(True)

plt.tight_layout()
plt.show()
plt.close()
#note4fig: fig2h_part1 fig2i_part1 fig2j_part1

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 12
color_md = "#4C78A8"
color_knee = "#F58518"

In [ ]:
matrix_md_path = "/Users/apple/Downloads/bacdrops/sal/run1.matrix.tsv"
matrix_knee_path = "/Users/apple/Downloads/bacdrops/sal/knee.matrix.tsv"
gff_path = "/Users/apple/Downloads/bacdrops/sal/28901.gff"
# Fixed-width bins by rRNA percentage (5% intervals)
bin_edges = np.arange(65, 101, 5)  # 65-70, 70-75, ..., 95-100
bin_min_cells = 20  # minimum cells per bin
bin_min_cells = 20  # minimum cells per bin
p_threshold = 0.01  # significance threshold for bin-mean corr

rrna_genes = set()
cds_genes = set()
gene_names = {}
with open(gff_path) as f:
    for line in f:
        if line.startswith("#") or not line.strip():
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 9:
            continue
        feature = cols[2]
        attrs_str = cols[8]
        attrs = {}
        for pair in attrs_str.split(";"):
            pair = pair.strip()
            if "=" in pair:
                k, v = pair.split("=", 1)
                attrs[k.strip()] = v.strip()
        if feature == "gene":
            gid = attrs.get("ID", "")
            if gid and attrs.get("gene_biotype") == "rRNA":
                rrna_genes.add(gid)
            # gene name
            gname = attrs.get("gene") or attrs.get("Name")
            if gid and gname:
                gene_names[gid] = gname
        elif feature == "CDS":
            # Use Parent as gene ID (consistent with gene feature ID)
            parent = attrs.get("Parent", "")
            if parent:
                cds_genes.add(parent)
# Also collect tRNA genes
trna_genes = set()
# Re-scan for tRNA features (they have gene_biotype=tRNA on gene lines,
# or feature type tRNA on the tRNA lines with Parent pointing to gene)
with open(gff_path) as f:
    for line in f:
        if line.startswith("#") or not line.strip():
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 9:
            continue
        feature = cols[2]
        attrs_str = cols[8]
        attrs = {}
        for pair in attrs_str.split(";"):
            pair = pair.strip()
            if "=" in pair:
                k, v = pair.split("=", 1)
                attrs[k.strip()] = v.strip()
        if feature == "gene" and attrs.get("gene_biotype") == "tRNA":
            gid = attrs.get("ID", "")
            if gid:
                trna_genes.add(gid)
        elif feature == "tRNA":
            parent = attrs.get("Parent", "")
            if parent:
                trna_genes.add(parent)
print(f"rRNA genes in GFF: {len(rrna_genes)}")
print(f"tRNA genes in GFF: {len(trna_genes)}")
print(f"CDS genes in GFF:  {len(cds_genes)}")
print(f"Gene names:         {len(gene_names)}")

In [ ]:
matrix_md = pd.read_csv(matrix_md_path, sep="\t", index_col=0)
print(f"MD matrix: {matrix_md.shape[0]} genes x {matrix_md.shape[1]} cells")
# rRNA percentage
matched_rrna_md = matrix_md.index.intersection(rrna_genes)
print(f"Matched rRNA: {len(matched_rrna_md)}")
rrna_counts_md = matrix_md.loc[matched_rrna_md].sum(axis=0)
total_counts_md = matrix_md.sum(axis=0)
rrna_pct_md = rrna_counts_md / total_counts_md
# Non-rRNA (mRNA) matrix
mrna_md = matrix_md.loc[~matrix_md.index.isin(rrna_genes) & ~matrix_md.index.isin(trna_genes)]
total_mrna_md = mrna_md.sum(axis=0)
valid_md = total_mrna_md > 0
mrna_md = mrna_md.loc[:, valid_md]
rrna_pct_md = rrna_pct_md.loc[valid_md]
total_mrna_md = total_mrna_md.loc[valid_md]
# CPM normalization by non-rRNA reads
mrna_cpm_md = mrna_md.div(total_mrna_md, axis=1) * 1e6
mrna_logcpm_md = np.log1p(mrna_cpm_md)
print(f"Valid cells (mRNA>0): {valid_md.sum()}")
matched_trna_md = mrna_md.index.intersection(trna_genes)
print(f"Matched tRNA (already excluded): {len(matched_trna_md)}")
print(f"rRNA%: median={rrna_pct_md.median():.3f}, range=[{rrna_pct_md.min():.3f}, {rrna_pct_md.max():.3f}]")

In [ ]:
matrix_knee = pd.read_csv(matrix_knee_path, sep="\t", index_col=0)
print(f"Knee matrix: {matrix_knee.shape[0]} genes x {matrix_knee.shape[1]} cells")
matched_rrna_knee = matrix_knee.index.intersection(rrna_genes)
print(f"Matched rRNA: {len(matched_rrna_knee)}")
rrna_counts_knee = matrix_knee.loc[matched_rrna_knee].sum(axis=0)
total_counts_knee = matrix_knee.sum(axis=0)
rrna_pct_knee = rrna_counts_knee / total_counts_knee
mrna_knee = matrix_knee.loc[~matrix_knee.index.isin(rrna_genes) & ~matrix_knee.index.isin(trna_genes)]
total_mrna_knee = mrna_knee.sum(axis=0)
valid_knee = total_mrna_knee > 0
mrna_knee = mrna_knee.loc[:, valid_knee]
rrna_pct_knee = rrna_pct_knee.loc[valid_knee]
total_mrna_knee = total_mrna_knee.loc[valid_knee]
mrna_cpm_knee = mrna_knee.div(total_mrna_knee, axis=1) * 1e6
mrna_logcpm_knee = np.log1p(mrna_cpm_knee)
print(f"Valid cells (mRNA>0): {valid_knee.sum()}")
matched_trna_knee = mrna_knee.index.intersection(trna_genes)
print(f"Matched tRNA (already excluded): {len(matched_trna_knee)}")
print(f"rRNA%: median={rrna_pct_knee.median():.3f}, range=[{rrna_pct_knee.min():.3f}, {rrna_pct_knee.max():.3f}]")

In [ ]:
def compute_bin_mean_corr(expr_df, rrna_pct_series, bin_edges, min_cells=20):
    """
    Fixed-width bins by rRNA%.
    For each gene, compute mean expression per bin.
    Then compute Spearman correlation between bin mean rRNA% and bin mean expression.
    Returns:
      corr_df: DataFrame with columns [gene, rho_bin, pvalue_bin, n_bins_used]
      bin_rrna_mean: Series of bin mean rRNA%
    """
    rrna_pct = rrna_pct_series * 100
    # Assign bins by fixed-width intervals
    bin_idx = np.digitize(rrna_pct.values, bin_edges) - 1
    bin_idx = np.clip(bin_idx, 0, len(bin_edges) - 2)
    cell_df = pd.DataFrame({"rrna_pct": rrna_pct.values, "bin": bin_idx}, index=rrna_pct.index)
    # Filter small bins
    bin_counts = cell_df["bin"].value_counts()
    valid_bins = bin_counts[bin_counts >= min_cells].index.sort_values()
    if len(valid_bins) < 4:
        return None, None
    cell_df = cell_df[cell_df["bin"].isin(valid_bins)]
    bin_rrna_mean = cell_df.groupby("bin")["rrna_pct"].mean()
    valid_cells = cell_df.index
    expr_valid = expr_df[valid_cells]
    results = []
    for gene in expr_valid.index:
        expr = expr_valid.loc[gene]
        bin_mean = expr.groupby(cell_df["bin"]).mean()
        bin_mean = bin_mean.loc[valid_bins].dropna()
        if len(bin_mean) < 4:
            continue
        rrna_vals = bin_rrna_mean.loc[bin_mean.index]
        rho, pval = spearmanr(rrna_vals, bin_mean.values)
        results.append({"gene": gene, "rho_bin": rho, "pvalue_bin": pval, "n_bins": len(bin_mean)})
    corr_df = pd.DataFrame(results)
    return corr_df, bin_rrna_mean
# Also compute cell-level spearman for comparison
def compute_cell_corr(expr_df, rrna_pct_series):
    results = []
    for gene in expr_df.index:
        expr = expr_df.loc[gene]
        rho, pval = spearmanr(rrna_pct_series, expr)
        results.append({"gene": gene, "rho_cell": rho, "pvalue_cell": pval})
    return pd.DataFrame(results)

In [ ]:
print("Computing ModuleDrops bin-mean correlation...")
p_threshold = 0.01
bin_corr_md, bin_rrna_md = compute_bin_mean_corr(
    mrna_logcpm_md, rrna_pct_md, bin_edges=bin_edges, min_cells=bin_min_cells
)
print(f"  Genes with valid bin-mean corr: {len(bin_corr_md)}")
sig_md = bin_corr_md[(bin_corr_md["pvalue_bin"] < p_threshold) & (bin_corr_md["rho_bin"] > 0)]
print(f"  Sig positive (p<{p_threshold}): {len(sig_md)}")
print("\nComputing Knee bin-mean correlation...")
bin_corr_knee, bin_rrna_knee = compute_bin_mean_corr(
    mrna_logcpm_knee, rrna_pct_knee, bin_edges=bin_edges, min_cells=bin_min_cells
)
print(f"  Genes with valid bin-mean corr: {len(bin_corr_knee)}")
sig_knee = bin_corr_knee[(bin_corr_knee["pvalue_bin"] < p_threshold) & (bin_corr_knee["rho_bin"] > 0)]
print(f"  Sig positive (p<{p_threshold}): {len(sig_knee)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5), dpi=300, gridspec_kw={"width_ratios": [1, 1.5]})
# Panel A: Number of sig positive genes
ax = axes[0]
methods = ["Knee", "ModuleDrops"]
counts = [len(sig_knee), len(sig_md)]
colors = [color_knee, color_md]
bars = ax.bar(methods, counts, width=0.55, color=colors, edgecolor="none")
ax.set_ylabel(f"Sig. positive genes\n(bin-mean, p<{p_threshold})")
ax.set_title("Number of detected genes")
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, n, str(n), ha="center", va="bottom", fontsize=10)
# Panel B: rho distribution of sig positive genes
ax = axes[1]
rho_data = [sig_knee["rho_bin"].values, sig_md["rho_bin"].values]
bp = ax.boxplot(rho_data, labels=methods, showfliers=False, patch_artist=True)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.35)
for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(1.2)
for w in bp["whiskers"]:
    w.set_color("black")
    w.set_linewidth(0.8)
for cap in bp["caps"]:
    cap.set_color("black")
    cap.set_linewidth(0.8)
# Jittered points
rng = np.random.default_rng(42)
for i, (vals, c) in enumerate(zip(rho_data, colors)):
    x = i + 1 + rng.uniform(-0.08, 0.08, len(vals))
    ax.scatter(x, vals, s=20, alpha=0.7, color=c, edgecolor="black", rasterized=True)
ax.set_ylabel("Spearman ρ (bin-mean)")
ax.set_title("Correlation strength")
for ax_i in axes:
    ax_i.spines["top"].set_visible(False)
    ax_i.spines["right"].set_visible(False)
    ax_i.tick_params(labelsize=9)
plt.tight_layout()
plt.show()
#note4fig: fig2e and fig2f

In [ ]:
# Supplementary: aspT / aspU / aspV (tRNA) bin-mean correlation
# Fixed-width bins by rRNA% (5% intervals), mean + 95% CI.
from statsmodels.nonparametric.smoothers_lowess import lowess
# aspT/U/V gene IDs (tRNA genes)
asp_genes = {}
for gid, gname in gene_names.items():
    if gname in ("aspT", "aspU", "aspV"):
        asp_genes[gname] = gid
print("asp tRNA genes found:")
for name, gid in asp_genes.items():
    print(f"  {name}: {gid}")
# Fixed-width bin edges (rRNA percentage)
bin_edges = np.arange(65, 101, 5)  # 65-70, 70-75, ..., 95-100
bin_labels = [f"{bin_edges[i]}-{bin_edges[i+1]}" for i in range(len(bin_edges) - 1)]
asp_colors = {"aspT": "#E45756", "aspU": "#72B7B2", "aspV": "#F58518"}
def plot_asp_binned(expr_full, rrna_pct_series, title="", ax=None):
    """Fixed-width bins, mean + 95% CI."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
    rrna_pct = rrna_pct_series * 100
    for gname, gid in asp_genes.items():
        if gid not in expr_full.index:
            continue
        expr = expr_full.loc[gid]
        bin_means = []
        bin_sems = []
        bin_midpoints = []
        bin_n = []
        for i in range(len(bin_edges) - 1):
            lo, hi = bin_edges[i], bin_edges[i + 1]
            mask = (rrna_pct >= lo) & (rrna_pct < hi)
            if i == len(bin_edges) - 2:  # last bin: include right edge
                mask = (rrna_pct >= lo) & (rrna_pct <= hi)
            vals = expr[mask].dropna()
            n = len(vals)
            if n < 5:
                continue
            mean_val = vals.mean()
            sem_val = vals.std() / np.sqrt(n)
            # 95% CI
            ci_val = 1.96 * sem_val
            bin_means.append(mean_val)
            bin_sems.append(ci_val)
            bin_midpoints.append((lo + hi) / 2)
            bin_n.append(n)
        if len(bin_means) < 3:
            continue
        color = asp_colors.get(gname, "gray")
        ax.errorbar(
            bin_midpoints,
            bin_means,
            yerr=bin_sems,
            fmt="o",
            ms=5,
            capsize=3,
            capthick=0.6,
            elinewidth=1,
            color=color,
            markeredgecolor="black",
            markeredgewidth=0.5,
            alpha=0.9,
            label=f"{gname}",
        )
        # LOWESS fit
        fit = lowess(bin_means, bin_midpoints, frac=1.0)
        ax.plot(fit[:, 0], fit[:, 1], color=color, lw=1.5, alpha=0.8)
        # Spearman
        rho, pval = spearmanr(bin_midpoints, bin_means)
        print(f"  {gname}: ρ={rho:.3f}, p={pval:.4f}")
    ax.set_xlabel("rRNA fraction (%)")
    ax.set_ylabel("Expression (log-normalized)")
    if title:
        ax.set_title(title, fontweight="bold")
    ax.set_xlim(65, 100)
    ax.set_xticks(np.arange(65, 101, 5))
    ax.legend(frameon=False, fontsize=9, loc="upper left")
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    ax.tick_params(labelsize=9)
    return ax

# Use raw matrix (including tRNA)
matrix_md_full = pd.read_csv(matrix_md_path, sep="\t", index_col=0)
matrix_knee_full = pd.read_csv(matrix_knee_path, sep="\t", index_col=0)
# Compute rrna_pct from full matrix
rrna_pct_md_full = matrix_md_full.loc[matrix_md_full.index.intersection(rrna_genes)].sum(axis=0) / matrix_md_full.sum(
    axis=0
)
rrna_pct_knee_full = matrix_knee_full.loc[matrix_knee_full.index.intersection(rrna_genes)].sum(
    axis=0
) / matrix_knee_full.sum(axis=0)
# CPM normalize (excluding rRNA but including tRNA)
mrna_md_full = matrix_md_full.loc[~matrix_md_full.index.isin(rrna_genes)]
total_mrna_md_full = mrna_md_full.sum(axis=0)
valid_md_full = total_mrna_md_full > 0
mrna_md_full = mrna_md_full.loc[:, valid_md_full]
rrna_pct_md_full = rrna_pct_md_full.loc[valid_md_full]
mrna_logcpm_md_full = np.log1p(mrna_md_full.div(total_mrna_md_full[valid_md_full], axis=1) * 1e6)
mrna_knee_full = matrix_knee_full.loc[~matrix_knee_full.index.isin(rrna_genes)]
total_mrna_knee_full = mrna_knee_full.sum(axis=0)
valid_knee_full = total_mrna_knee_full > 0
mrna_knee_full = mrna_knee_full.loc[:, valid_knee_full]
rrna_pct_knee_full = rrna_pct_knee_full.loc[valid_knee_full]
mrna_logcpm_knee_full = np.log1p(mrna_knee_full.div(total_mrna_knee_full[valid_knee_full], axis=1) * 1e6)
# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300)
print("ModuleDrops:")
plot_asp_binned(mrna_logcpm_md_full, rrna_pct_md_full, title="ModuleDrops", ax=axes[0])
print("\nKnee:")
plot_asp_binned(mrna_logcpm_knee_full, rrna_pct_knee_full, title="Knee", ax=axes[1])
plt.tight_layout()
plt.show()
#note4fig: fig2g (ModuleDrops fig was extracted and used)